In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

matches = pd.read_csv('clean_matches.csv')
deliveries = pd.read_csv('clean_deliveries.csv')

matches.columns = matches.columns.str.lower().str.strip()
deliveries.columns = deliveries.columns.str.lower().str.strip()

matches['matchid'] = matches['matchid'].astype(str).str.strip()
deliveries['matchid'] = deliveries['matchid'].astype(str).str.strip()

if 'date' in matches.columns:
    matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

merge_cols = [c for c in ['matchid','venue','season','date','team1','team2','winner'] if c in matches.columns]
deliveries = deliveries.merge(matches[merge_cols], on='matchid', how='left')

for col, default in [
    ('venue', 'Unknown'),
    ('season', 'Unknown'),
    ('team1', 'Unknown'),
    ('team2', 'Unknown'),
    ('winner', 'No Result')
]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(default)
    else:
        deliveries[col] = default

if 'date' in deliveries.columns:
    deliveries['date'] = pd.to_datetime(deliveries['date'], errors='coerce')
else:
    deliveries['date'] = pd.NaT

In [2]:
valid = deliveries[deliveries.get('is_valid_ball', 0) == 1]

innings = valid.groupby(['matchid', 'inning', 'batsman']).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

info_cols = ['matchid', 'inning', 'batsman']
for c in ['batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2']:
    if c in deliveries.columns:
        info_cols.append(c)

info = deliveries[info_cols].drop_duplicates(subset=['matchid', 'inning', 'batsman'])

innings = innings.merge(info, on=['matchid', 'inning', 'batsman'], how='left')

innings['strike_rate'] = np.where(
    innings['balls_faced'] > 0,
    innings['runs_scored'] / innings['balls_faced'] * 100,
    0
)

if 'date' in innings.columns:
    innings['date'] = pd.to_datetime(innings['date'], errors='coerce')
    innings = innings.sort_values(['batsman', 'date']).reset_index(drop=True)
else:
    innings = innings.sort_values(['batsman', 'matchid']).reset_index(drop=True)

In [3]:
innings['innings_played'] = innings.groupby('batsman').cumcount() + 1

innings['batsman_avg'] = innings.groupby('batsman')['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings['runs_scored'].mean())
)

innings['balls_faced_avg'] = innings.groupby('batsman')['balls_faced'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings['balls_faced'].mean())
)

cum_r = innings.groupby('batsman')['runs_scored'].cumsum().shift(1).fillna(0)
cum_b = innings.groupby('batsman')['balls_faced'].cumsum().shift(1).fillna(0)
innings['historical_sr'] = np.where(cum_b > 0, cum_r / cum_b * 100, 0)

temp = innings.sort_values(['batsman', 'bowling_team', 'matchid'])
temp['avg_vs_opp'] = temp.groupby(['batsman', 'bowling_team'])['runs_scored'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings['runs_scored'].mean())
)
innings = temp.sort_values(['batsman', 'matchid']).reset_index(drop=True)

In [4]:
innings_tot = deliveries.groupby(['matchid', 'inning'])['total_runs'].sum().reset_index(name='innings_total')

innings_tot = innings_tot.merge(
    deliveries[['matchid', 'inning', 'bowling_team']].drop_duplicates(),
    on=['matchid', 'inning']
)

innings_tot['opp_strength'] = innings_tot.groupby('bowling_team')['innings_total'].transform(
    lambda x: x.shift(1).expanding().mean().fillna(innings_tot['innings_total'].mean())
)

innings = innings.merge(
    innings_tot[['matchid', 'inning', 'opp_strength']],
    on=['matchid', 'inning'],
    how='left'
)

innings['opp_strength'] = innings['opp_strength'].fillna(innings['opp_strength'].mean())

if 'team1' in innings.columns:
    innings['home_advantage'] = (innings['batting_team'] == innings['team1']).astype(int)
else:
    innings['home_advantage'] = 0

def extract_year(val):
    if pd.isna(val):
        return 2020
    val = str(val).strip()
    if '/' in val:
        parts = val.split('/')
        if len(parts) == 2 and parts[1].isdigit():
            return int(parts[1])
        if parts[0].isdigit():
            return int(parts[0])
    digits = ''.join(c for c in val if c.isdigit())
    if len(digits) >= 4:
        return int(digits[-4:])
    return 2020

innings['season_year'] = innings['season'].apply(extract_year)

if 'date' in innings.columns and innings['date'].notna().any():
    mask = innings['season_year'] == 2020
    innings.loc[mask, 'season_year'] = innings.loc[mask, 'date'].dt.year

innings['season_year'] = innings['season_year'].astype(int)

le = LabelEncoder()
innings['venue_encoded'] = le.fit_transform(innings['venue'])

final = innings[innings['innings_played'] > 5].copy()

final.to_csv('processed_player_data.csv', index=False)
import joblib
joblib.dump(le, 'venue_encoder.pkl')

print("\nSaved processed_player_data.csv and venue_encoder.pkl")
print("Final rows:", len(final))
print("Final columns:", final.columns.tolist())
print("\nseason_year unique values:", sorted(final['season_year'].unique()))
print("\nSample (key columns):\n")
print(final[['batsman','runs_scored','batsman_avg','historical_sr','avg_vs_opp','venue_encoded','opp_strength','home_advantage','season_year']].head(5))


Saved processed_player_data.csv and venue_encoder.pkl
Final rows: 694
Final columns: ['matchid', 'inning', 'batsman', 'runs_scored', 'balls_faced', 'batting_team', 'bowling_team', 'venue', 'season', 'date', 'team1', 'team2', 'strike_rate', 'innings_played', 'batsman_avg', 'balls_faced_avg', 'historical_sr', 'avg_vs_opp', 'opp_strength', 'home_advantage', 'season_year', 'venue_encoded']

season_year unique values: [np.int64(2020)]

Sample (key columns):

       batsman  runs_scored  batsman_avg  historical_sr  avg_vs_opp  \
5     A Chopra         11.0          8.4      80.769231    24.00000   
14    A Kumble          1.0          2.8      63.636364    18.98943   
15    A Kumble          5.0          2.5      65.217391     1.50000   
34  AB Agarkar          5.0          9.8     113.953488    20.00000   
35  AB Agarkar          7.0          9.0     120.000000    18.98943   

    venue_encoded  opp_strength  home_advantage  season_year  
5               0    164.714286               0    